## lakeLoom Real Upload Trigger

This notebook triggers a real signed audio upload against the live dev lakeLoom app using the same authentication pattern as `pairing-api-test`:

1. Discover the dev app URL
2. Acquire an Xcode SPN OAuth token
3. Request a QR pairing token
4. Generate an ECDSA P-256 device keypair
5. Confirm pairing with the signed canonical request
6. Create a project
7. Assign the paired device to the project
8. Create a capture session
9. Upload a signed multipart audio payload with `client_ts`, `client_filename`, and `sha256_hex`

Run the cells top-to-bottom. The notebook is self-contained and does not require chat context.

In [0]:
%pip install --upgrade databricks-sdk cryptography
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
import json
import time
import uuid
import base64
import hashlib
import io
import struct
import wave
from dataclasses import dataclass

import requests
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec

wc = WorkspaceClient()
WORKSPACE_HOST = wc.config.host.rstrip('/')

SCOPE = 'lakeloom_credentials'
XCODE_CLIENT_ID = dbutils.secrets.get(SCOPE, 'xcode_client_id_dev_matthew_giglia_lakeloom')
XCODE_CLIENT_SECRET = dbutils.secrets.get(SCOPE, 'xcode_client_secret_dev_matthew_giglia_lakeloom')

APP_NAME = 'lakeloom-ai-dev'
APP_URL = None
for app in wc.apps.list():
    if app.name == APP_NAME:
        APP_URL = app.url.rstrip('/')
        break

if not APP_URL:
    raise ValueError(f'Could not find app URL for {APP_NAME}')

print('Workspace host:', WORKSPACE_HOST)
print('App name:      ', APP_NAME)
print('App url:       ', APP_URL)
print('Client id:     ', XCODE_CLIENT_ID)

run_context = {
    'started_at_unix': int(time.time()),
    'app_name': APP_NAME,
    'app_url': APP_URL,
}

In [0]:
token_url = f'{WORKSPACE_HOST}/oidc/v1/token'

token_resp = requests.post(
    token_url,
    data={
        'grant_type': 'client_credentials',
        'scope': 'all-apis',
        'client_id': XCODE_CLIENT_ID,
        'client_secret': XCODE_CLIENT_SECRET,
    },
    timeout=30,
)

token_resp.raise_for_status()
token_data = token_resp.json()
SPN_TOKEN = token_data['access_token']

print('Token endpoint:', token_url)
print('HTTP status:   ', token_resp.status_code)
print('Token type:    ', token_data.get('token_type'))
print('Expires in:    ', token_data.get('expires_in'))
print('Access token:  ', f"{SPN_TOKEN[:24]}...")

In [0]:
@dataclass
class SignedRequest:
    timestamp: str
    signature: str
    canonical: str
    body_sha256: str


def sign_request(private_key, method: str, path: str, body_bytes: bytes | None = None) -> SignedRequest:
    timestamp = str(int(time.time()))
    payload = body_bytes if body_bytes is not None else b''
    body_sha256 = hashlib.sha256(payload).hexdigest()
    canonical = f'{method}\n{path}\n{timestamp}\n{body_sha256}'
    signature_der = private_key.sign(canonical.encode(), ec.ECDSA(hashes.SHA256()))
    signature = base64.urlsafe_b64encode(signature_der).decode().rstrip('=')
    return SignedRequest(timestamp=timestamp, signature=signature, canonical=canonical, body_sha256=body_sha256)


def compact_json_bytes(payload: dict) -> bytes:
    return json.dumps(payload, separators=(',', ':')).encode()


def make_signed_json_headers(session_token: str, signed: SignedRequest) -> dict:
    return {
        'Authorization': f'Bearer {SPN_TOKEN}',
        'Content-Type': 'application/json',
        'X-Lakeloom-Session-Token': session_token,
        'X-Lakeloom-Timestamp': signed.timestamp,
        'X-Lakeloom-Signature': signed.signature,
    }


def build_test_wav_bytes(sample_rate: int = 16000, frame_count: int = 1600) -> bytes:
    wav_buffer = io.BytesIO()
    with wave.open(wav_buffer, 'wb') as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(sample_rate)
        frames = b''.join(struct.pack('<h', 0) for _ in range(frame_count))
        wav_file.writeframes(frames)
    return wav_buffer.getvalue()


def build_multipart_body(fields: dict[str, str], file_field_name: str, filename: str, mime_type: str, file_bytes: bytes):
    boundary = f'----lakeloom{uuid.uuid4().hex}'
    parts: list[bytes] = []

    for name, value in fields.items():
        parts.append(f'--{boundary}\r\n'.encode())
        parts.append(f'Content-Disposition: form-data; name="{name}"\r\n\r\n'.encode())
        parts.append(str(value).encode())
        parts.append(b'\r\n')

    parts.append(f'--{boundary}\r\n'.encode())
    parts.append(f'Content-Disposition: form-data; name="{file_field_name}"; filename="{filename}"\r\n'.encode())
    parts.append(f'Content-Type: {mime_type}\r\n\r\n'.encode())
    parts.append(file_bytes)
    parts.append(b'\r\n')
    parts.append(f'--{boundary}--\r\n'.encode())

    return boundary, b''.join(parts)


print('Helpers loaded.')

In [0]:
qr_resp = requests.get(
    f'{APP_URL}/api/pairing/qr',
    headers={'Authorization': f'Bearer {SPN_TOKEN}'},
    timeout=30,
    allow_redirects=False,
)
qr_resp.raise_for_status()
qr_data = qr_resp.json()
SESSION_TOKEN = qr_data['session']['token']

PRIVATE_KEY = ec.generate_private_key(ec.SECP256R1(), default_backend())
PUBLIC_KEY = PRIVATE_KEY.public_key()
PUBLIC_KEY_SPKI_DER = PUBLIC_KEY.public_bytes(
    serialization.Encoding.DER,
    serialization.PublicFormat.SubjectPublicKeyInfo,
)
DEVICE_PUBKEY_B64URL = base64.urlsafe_b64encode(PUBLIC_KEY_SPKI_DER).decode().rstrip('=')

print('QR endpoint status: ', qr_resp.status_code)
print('Session token:      ', f"{SESSION_TOKEN[:24]}...")
print('Device key bytes:   ', len(PUBLIC_KEY_SPKI_DER))
print('QR payload keys:    ', list(qr_data.keys()))

In [0]:
pair_body = {
    'device_pubkey': DEVICE_PUBKEY_B64URL,
    'device_label': f'Notebook Upload Trigger {int(time.time())}',
}
pair_body_bytes = compact_json_bytes(pair_body)
pair_path = '/api/pairing/confirm'
pair_signed = sign_request(PRIVATE_KEY, 'POST', pair_path, pair_body_bytes)
pair_headers = make_signed_json_headers(SESSION_TOKEN, pair_signed)

pair_resp = requests.post(
    f'{APP_URL}{pair_path}',
    headers=pair_headers,
    data=pair_body_bytes,
    timeout=30,
    allow_redirects=False,
)

print('HTTP status:   ', pair_resp.status_code)
print('Canonical hash:', pair_signed.body_sha256)
print('Response text: ', pair_resp.text[:400])

pair_resp.raise_for_status()
pair_data = pair_resp.json()
PAIRED_SESSION_ID = pair_data['paired_session_id']
print('Paired session:', PAIRED_SESSION_ID)

In [0]:
project_body = {
    'name': f'Upload Trigger Project {int(time.time())}',
    'description': 'Temporary project created from notebook to trigger upload telemetry',
    'workspace_id': WORKSPACE_HOST,
    'client_generated_id': str(uuid.uuid4()),
}
project_body_bytes = compact_json_bytes(project_body)
project_path = '/api/v1/projects'
project_signed = sign_request(PRIVATE_KEY, 'POST', project_path, project_body_bytes)
project_headers = make_signed_json_headers(SESSION_TOKEN, project_signed)

project_resp = requests.post(
    f'{APP_URL}{project_path}',
    headers=project_headers,
    data=project_body_bytes,
    timeout=30,
    allow_redirects=False,
)

print('HTTP status: ', project_resp.status_code)
print('Response:    ', project_resp.text[:500])

project_resp.raise_for_status()
try:
    project_data = project_resp.json()
except ValueError as e:
    raise RuntimeError(f'Project creation did not return JSON: {project_resp.text[:500]}') from e


def extract_project_id(payload):
    if isinstance(payload, dict):
        for id_key in ('id', 'project_id'):
            project_id = payload.get(id_key)
            if isinstance(project_id, str) and project_id:
                return project_id
        for key in ('project', 'data', 'result', 'item'):
            nested = payload.get(key)
            if nested is not None:
                nested_id = extract_project_id(nested)
                if nested_id:
                    return nested_id
        for key in ('projects', 'items', 'results'):
            nested = payload.get(key)
            if isinstance(nested, list):
                for item in nested:
                    nested_id = extract_project_id(item)
                    if nested_id:
                        return nested_id
    elif isinstance(payload, list):
        for item in payload:
            nested_id = extract_project_id(item)
            if nested_id:
                return nested_id
    return None


PROJECT_ID = extract_project_id(project_data)
if not PROJECT_ID:
    raise RuntimeError(
        'Project creation succeeded but no project id was found in response: '
        f'{json.dumps(project_data)[:1000]}'
    )

print('Project id:  ', PROJECT_ID)

In [0]:
assign_body = {'paired_session_id': PAIRED_SESSION_ID}
assign_body_bytes = compact_json_bytes(assign_body)
assign_path = f'/api/v1/projects/{PROJECT_ID}/devices'
assign_signed = sign_request(PRIVATE_KEY, 'POST', assign_path, assign_body_bytes)
assign_headers = make_signed_json_headers(SESSION_TOKEN, assign_signed)

assign_resp = requests.post(
    f'{APP_URL}{assign_path}',
    headers=assign_headers,
    data=assign_body_bytes,
    timeout=30,
    allow_redirects=False,
)

print('HTTP status: ', assign_resp.status_code)
print('Response:    ', assign_resp.text[:500])

assign_resp.raise_for_status()
assign_data = assign_resp.json()
print('Assignment id:', assign_data.get('id'))

In [0]:
capture_body = {'label': f'Upload Trigger Capture {int(time.time())}'}
capture_body_bytes = compact_json_bytes(capture_body)
capture_path = f'/api/projects/{PROJECT_ID}/captures'
capture_signed = sign_request(PRIVATE_KEY, 'POST', capture_path, capture_body_bytes)
capture_headers = make_signed_json_headers(SESSION_TOKEN, capture_signed)

capture_resp = requests.post(
    f'{APP_URL}{capture_path}',
    headers=capture_headers,
    data=capture_body_bytes,
    timeout=30,
    allow_redirects=False,
)

print('HTTP status: ', capture_resp.status_code)
print('Response:    ', capture_resp.text[:500])

capture_resp.raise_for_status()
capture_data = capture_resp.json()
CAPTURE_ID = capture_data['id']
print('Capture id:  ', CAPTURE_ID)

In [0]:
volume_base = '/Volumes/hls_fde_dev/dev_matthew_giglia_lakeloom/session_audio'
volume_directory = f"{volume_base}/{PROJECT_ID}/{CAPTURE_ID}"
candidate_paths = [volume_directory, f"dbfs:{volume_directory}"]

from databricks.sdk import WorkspaceClient

sdk_wc = WorkspaceClient()
files_api = sdk_wc.files

print('Volume directory:', volume_directory)
print('Candidate paths: ', candidate_paths)
print('Files API methods:', [name for name in dir(files_api) if 'directory' in name.lower()])

for candidate_path in candidate_paths:
    print(f'\nTesting create_directory on: {candidate_path}')
    try:
        result = files_api.create_directory(candidate_path)
        print('create_directory result:', result)
    except Exception as e:
        print('create_directory error:', type(e).__name__, str(e))
        print('error attrs:', {
            'status_code': getattr(e, 'status_code', getattr(e, 'statusCode', None)),
            'error_code': getattr(e, 'error_code', getattr(e, 'errorCode', None)),
            'details': getattr(e, 'details', None),
            'args': getattr(e, 'args', None),
        })

for candidate_path in candidate_paths:
    print(f'\nTesting dbutils.fs.mkdirs on: {candidate_path}')
    try:
        result = dbutils.fs.mkdirs(candidate_path)
        print('dbutils.fs.mkdirs result:', result)
    except Exception as e:
        print('dbutils.fs.mkdirs error:', type(e).__name__, str(e))

for candidate_path in candidate_paths:
    print(f'\nTesting dbutils.fs.ls on: {candidate_path}')
    try:
        listing = dbutils.fs.ls(candidate_path)
        print('dbutils.fs.ls entries:', len(listing))
        display(listing[:5])
    except Exception as e:
        print('dbutils.fs.ls error:', type(e).__name__, str(e))

In [0]:
AUDIO_BYTES = build_test_wav_bytes()
AUDIO_FILENAME = 'trigger.wav'
AUDIO_MIME_TYPE = 'audio/wav'
AUDIO_SHA256 = hashlib.sha256(AUDIO_BYTES).hexdigest()
CLIENT_TS = str(int(time.time()))

multipart_fields = {
    'client_ts': CLIENT_TS,
    'client_filename': AUDIO_FILENAME,
    'sha256_hex': AUDIO_SHA256,
}

UPLOAD_BOUNDARY, UPLOAD_BODY = build_multipart_body(
    fields=multipart_fields,
    file_field_name='file',
    filename=AUDIO_FILENAME,
    mime_type=AUDIO_MIME_TYPE,
    file_bytes=AUDIO_BYTES,
)

upload_path = f'/api/captures/{CAPTURE_ID}/audio'
upload_signed = sign_request(PRIVATE_KEY, 'POST', upload_path, UPLOAD_BODY)
upload_headers = {
    'Authorization': f'Bearer {SPN_TOKEN}',
    'Content-Type': f'multipart/form-data; boundary={UPLOAD_BOUNDARY}',
    'X-Lakeloom-Session-Token': SESSION_TOKEN,
    'X-Lakeloom-Timestamp': upload_signed.timestamp,
    'X-Lakeloom-Signature': upload_signed.signature,
}

upload_resp = requests.post(
    f'{APP_URL}{upload_path}',
    headers=upload_headers,
    data=UPLOAD_BODY,
    timeout=60,
    allow_redirects=False,
)

print('HTTP status: ', upload_resp.status_code)
print('Response:    ', upload_resp.text[:1200])

upload_resp.raise_for_status()
UPLOAD_DATA = upload_resp.json()
print('Upload id:   ', UPLOAD_DATA.get('id'))
print('Kind:        ', UPLOAD_DATA.get('kind'))
print('Size bytes:  ', UPLOAD_DATA.get('size_bytes'))
print('Client ts:   ', UPLOAD_DATA.get('client_ts'))
print('TS source:   ', UPLOAD_DATA.get('client_ts_source'))

In [0]:
summary = {
    'app_url': APP_URL,
    'paired_session_id': PAIRED_SESSION_ID,
    'project_id': PROJECT_ID,
    'capture_id': CAPTURE_ID,
    'upload_id': UPLOAD_DATA.get('id'),
    'upload_kind': UPLOAD_DATA.get('kind'),
    'volume_path': UPLOAD_DATA.get('volume_path'),
    'size_bytes': UPLOAD_DATA.get('size_bytes'),
    'sha256_hex': UPLOAD_DATA.get('sha256_hex'),
    'client_ts': UPLOAD_DATA.get('client_ts'),
    'client_ts_source': UPLOAD_DATA.get('client_ts_source'),
}

print(json.dumps(summary, indent=2))